# Protocol 67 Serve Model From Colab

This notebook loads the trained Protocol 67 LoRA adapter in Colab and exposes it as a temporary HTTP API. Use this when the trained model is too large or slow to run locally.

Expected model adapter location after restoring from Google Drive:

```text
model_service/outputs/llama3b-slang-lora
```

The API created by this notebook is temporary. The public URL changes whenever the Colab runtime or tunnel restarts.


## Adapter Folder Structure

The inference code expects the trained adapter to end up here inside the cloned repository:

```text
model_service/outputs/llama3b-slang-lora
```

Keep the adapter files directly inside that folder, for example `adapter_config.json`, `adapter_model.safetensors`, and `tokenizer.json`. Avoid adding an extra nested `llama3b-slang-lora` folder inside it.

By default, this notebook restores the adapter from:

```text
`/content/drive/MyDrive/protocol67-models/llama3b-slang-lora` by default, or the configured `DRIVE_ADAPTER_DIR` if you change it
```

You can use your own Drive path or adapter folder name, but update the restore command and `ADAPTER_DIR` in `model_service/src/translator.py` if your final adapter location changes.


In [ ]:
# Change this if your trained adapter is stored somewhere else in Google Drive.
DRIVE_MODEL_DIR = "/content/drive/MyDrive/protocol67-models"
ADAPTER_NAME = "llama3b-slang-lora"
DRIVE_ADAPTER_DIR = f"{DRIVE_MODEL_DIR}/{ADAPTER_NAME}"
DRIVE_ADAPTER_DIR


## 1. Set Colab Runtime Type

Before running the cells, switch Colab to a GPU runtime:

```text
Runtime -> Change runtime type -> Hardware accelerator -> GPU
```

After the runtime reconnects, run `nvidia-smi` to confirm that a GPU is available.


In [ ]:
!nvidia-smi


## 2. Clone The Repository

Clone the project into the Colab runtime and move into the repository root. Later commands assume the current folder contains `backend`, `data_pipeline`, `frontend`, and `model_service`.


In [ ]:
%cd /content
!git clone https://github.com/hsyen78444/Protocol-67.git
%cd /content/Protocol-67
!ls


## 3. Install Dependencies

Install the model dependencies plus FastAPI, Uvicorn, ngrok, and Colab async-loop support for the temporary API server.


In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -r model_service/requirements.txt
!python -m pip install -U bitsandbytes torchao fastapi uvicorn pyngrok nest_asyncio


## 4. Authenticate With Hugging Face

The base model is `meta-llama/Llama-3.2-3B-Instruct`, which is gated. Before running the login cell:

1. Create or sign in to a Hugging Face account: https://huggingface.co/join
2. Generate a read token: https://huggingface.co/settings/tokens
3. Request access for the Llama model: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

Use a token with `Read` permission. When prompted, paste the token into the hidden input. If asked whether to add the token as a git credential, choose `N` unless you specifically need git credential storage.


In [ ]:
!hf auth login
!hf auth whoami


## 5. Restore The Trained Adapter From Google Drive

Mount Google Drive and copy the trained adapter folder back into the repository. This assumes the training notebook saved the adapter at:

```text
`/content/drive/MyDrive/protocol67-models/llama3b-slang-lora` by default, or the configured `DRIVE_ADAPTER_DIR` if you change it
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p model_service/outputs
!cp -r "{DRIVE_ADAPTER_DIR}" model_service/outputs/
!ls model_service/outputs/llama3b-slang-lora


## 6. Verify Model Access

Run a small tokenizer access check before loading the full model. If this fails with a gated repository error, verify that the Hugging Face account has Llama access and that the active token belongs to that account.


In [ ]:
!python -c "from transformers import AutoTokenizer; AutoTokenizer.from_pretrained('meta-llama/Llama-3.2-3B-Instruct'); print('access works')"


## 7. Load And Warm Up The Translator

Import the project translator and run one sample translation. The first call loads the base model and LoRA adapter, so it can take a while.


In [ ]:
from model_service.src.translator import translate

translate("bro is cooked fr")


## 8. Create The FastAPI App

This creates a small `/translate` endpoint that accepts JSON shaped like `{"text": "..."}` and returns the model translator output.


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Protocol 67 Colab Model API")

class TranslateRequest(BaseModel):
    text: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/translate")
def translate_endpoint(request: TranslateRequest):
    return translate(request.text)


## 9. Start The Public Tunnel

Create an ngrok account and get an authtoken here: https://dashboard.ngrok.com/get-started/your-authtoken

Paste the token into the hidden prompt. The notebook starts Uvicorn in a background thread and prints the public API URL.


In [ ]:
from getpass import getpass
from pyngrok import ngrok
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

ngrok.set_auth_token(getpass("Paste ngrok authtoken: "))
public_url = ngrok.connect(8001).public_url
print("Public base URL:", public_url)
print("Translate endpoint:", public_url + "/translate")

config = uvicorn.Config(app, host="0.0.0.0", port=8001, log_level="info")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()


## 10. Test The Public API

Replace `PUBLIC_BASE_URL` with the ngrok base URL printed above. This calls the Colab-hosted model through HTTP.


In [ ]:
import requests

PUBLIC_BASE_URL = "PASTE_PUBLIC_BASE_URL_HERE"
response = requests.post(
    PUBLIC_BASE_URL.rstrip("/") + "/translate",
    json={"text": "bro is cooked fr"},
    timeout=120,
)
response.raise_for_status()
response.json()


## 11. Use The URL Locally

The public endpoint can be called from your local backend or tested directly from PowerShell:

```powershell
$body = @{text = "bro is cooked fr"} | ConvertTo-Json
Invoke-RestMethod -Uri "https://YOUR-NGROK-URL.ngrok-free.app/translate" `
  -Method POST `
  -Body $body `
  -ContentType "application/json"
```

The ngrok URL is temporary and changes when the tunnel restarts.
